In [ ]:
!mkdir crudos
!mkdir procesados
!mkdir documentos
!mkdir modelos
print("CARPETAS CREADAS")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Scikit-learn: librería de Machine Learning
from sklearn.dummy import DummyRegressor       # Para el baseline
from sklearn.linear_model import LinearRegression  # Modelo simple
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
print("LIBRERIAS IMPORTADAS")

In [ ]:
BASE_DIR = Path("/content/")
DATA_RAW = BASE_DIR / "crudos"
DATA_PROCESSED = BASE_DIR / "procesados"
DATA_DOCS = BASE_DIR / "documentos"
DATA_MODELS = BASE_DIR / "modelos"
print("CARPETAS CREADAS!!")

In [ ]:
limpio = DATA_RAW / "dataset_limpio.csv"
df = pd.read_csv(limpio)
print(df.columns.tolist())
print(f"Filas: {len(df)}")

In [ ]:
# VERIFICAR CALIDAD DEL DATASET
print("INFORMACIÓN DEL DATASET:")
print("=" * 50)
df.info()

print("\nVALORES NULOS POR COLUMNA:")
print(df.isnull().sum())

print(f"\nRANGO TEMPORAL:")
print(f"  Desde: {df['fecha_pedido'].min()}")
print(f"  Hasta: {df['fecha_pedido'].max()}")


In [ ]:
#CREACION DE DF UNICAMENTE CON LOS DATOS DE FECHA Y DE VENTAS POR DIA
df['fecha_pedido'] = pd.to_datetime(df['fecha_pedido'])
# Agrupar por fecha y sumar los ingresos
df_diario = df.groupby('fecha_pedido')['precio_pedido'].sum().reset_index()
# Renombrar la columna para reflejar que ahora es un total
df_diario.rename(columns={'precio_pedido': 'ventas_por_dia'}, inplace=True)
# Ver el resultado
print(df_diario.head())

In [ ]:
# CREACIÓN DE DF DIARIO CON VENTAS Y FESTIVAL
df['fecha_pedido'] = pd.to_datetime(df['fecha_pedido'])

# Convertir 'festival' de texto (Yes/No) a número (1/0)
df['festival'] = df['festival'].map({'Yes': 1, 'No': 0})

# Agrupar por fecha: sumar ingresos y marcar si hubo festival ese día (max devuelve 1 si hubo)
df_diario = df.groupby('fecha_pedido').agg(
    ventas_por_dia=('precio_pedido', 'sum'),
    festival=('festival', 'max')
).reset_index()

# Ordenar por fecha por seguridad antes de crear los lags
df_diario = df_diario.sort_values('fecha_pedido').reset_index(drop=True)

print(df_diario.head())
print(f"\nDías totales: {len(df_diario)} | Días con festival: {df_diario['festival'].sum()}")

In [ ]:
# COMPARAR VENTAS CON Y SIN FESTIVAL
promedio_fest = df_diario.groupby("festival")["ventas_por_dia"].mean()

print("VENTAS PROMEDIO POR FESTIVAL:")
print("=" * 50)
print(f"   Sin festival (0): ${promedio_fest[0]:,.2f}")
print(f"   Con festival (1): ${promedio_fest[1]:,.2f}")
print(f"\nDIFERENCIA: ${promedio_fest[1] - promedio_fest[0]:,.2f}")
print(f"IMPACTO: {((promedio_fest[1] / promedio_fest[0]) - 1) * 100:.1f}%")

# Gráfico de barras comparativo
fig, ax = plt.subplots(figsize=(8, 5))
colores = ['#95a5a6', '#e74c3c']
barras = ax.bar(['Sin festival', 'Con festival'],
                [promedio_fest[0], promedio_fest[1]],
                color=colores, edgecolor='#2c3e50', linewidth=1.5)

for barra, valor in zip(barras, [promedio_fest[0], promedio_fest[1]]):
    ax.text(barra.get_x() + barra.get_width()/2., barra.get_height(),
            f'${valor:,.2f}', ha='center', va='bottom',
            fontsize=12, fontweight='bold')

ax.set_title('Ventas Promedio: Con vs Sin Festival',
             fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Ventas promedio ($)', fontsize=11)
ax.grid(True, alpha=0.3, axis='y', linestyle='--')

plt.tight_layout()
plt.show()

In [ ]:
# GRÁFICO 1: Serie histórica de ventas
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(df_diario["fecha_pedido"], df_diario["ventas_por_dia"],
        color='#2c7bb6', linewidth=1.5, alpha=0.8)

# Línea de tendencia (media móvil de 30 días)
df_diario['tendencia_30'] = df_diario['ventas_por_dia'].rolling(window=30).mean()
ax.plot(df_diario["fecha_pedido"], df_diario['tendencia_30'],
        color='#e74c3c', linewidth=2.5, linestyle='--', label='Tendencia (30 días)')

ax.set_title(f'Ventas Diarias Históricas ({len(df_diario)} días)',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Fecha', fontsize=11)
ax.set_ylabel('Ventas ($)', fontsize=11)
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(loc='upper left')

plt.tight_layout()
plt.show()

print("\nOBSERVACIONES DE LA SERIE:")
print(f"   • Ventas mínimas: ${df_diario['ventas_por_dia'].min():,.2f}")
print(f"   • Ventas máximas: ${df_diario['ventas_por_dia'].max():,.2f}")
print(f"   • Ventas promedio: ${df_diario['ventas_por_dia'].mean():,.2f}")

In [ ]:
# CREAR VARIABLES DE REZAGO (LAG)
# shift(1) mueve los valores una posición hacia abajo
# Es decir: lag_1 en la fila 10 = ventas_dia de la fila 9

df_diario["lag_1"] = df_diario["ventas_por_dia"].shift(1)   # Ventas de ayer
df_diario["lag_7"] = df_diario["ventas_por_dia"].shift(7)   # Ventas de hace 7 días

print("VARIABLES DE REZAGO CREADAS:")
print("=" * 70)
print(df_diario[["fecha_pedido", "ventas_por_dia", "lag_1", "lag_7"]].head(10).to_string())

print("\nOBSERVEN: Las primeras filas tienen NaN en lag_1 y lag_7")
print("   Porque no hay datos anteriores disponibles")

In [ ]:
# CREAR MEDIA MÓVIL SIN FUGA
# shift(1) para excluir el día actual
# rolling(7) para tomar los últimos 7 días
# mean() para promediar

df_diario["media_movil_7"] = (
    df_diario["ventas_por_dia"]
    .shift(1)          # EXCLUIR EL DÍA ACTUAL (evita fuga)
    .rolling(window=7) # Últimos 7 días
    .mean()            # Promedio
)

print("MEDIA MÓVIL DE 7 DÍAS (SIN FUGA):")
print("=" * 80)
print(df_diario[["fecha_pedido", "ventas_por_dia", "lag_1", "media_movil_7"]].head(12).to_string())

print("\nVERIFICACIÓN DE NO FUGA:")
print("   • media_movil_7 en la fila 10 = promedio de ventas_dia de filas 3-9")
print("   • NO incluye ventas_dia de fila 10 (el día actual)")
print("   • Por lo tanto, está disponible antes de predecir")

In [ ]:
# ELIMINAR FILAS CON NaN (solo en las variables de rezago)
print(f"Filas ANTES de eliminar NaN: {len(df_diario)}")

df_modelo = df_diario.dropna(subset=["lag_1", "lag_7", "media_movil_7"]).copy()

print(f"Filas DESPUÉS de eliminar NaN: {len(df_modelo)}")
print(f"Filas eliminadas: {len(df_diario) - len(df_modelo)}")
print(f"\nLas filas eliminadas son las primeras que no tienen historial de 7 días")

print(f"\nPrimeras 3 filas del dataset de modelado:")
print(df_modelo.head(3).to_string())

In [ ]:
#DEFINIR VARIABLES PREDICTORAS (X) Y OBJETIVO (y)
df_modelo['dia_semana'] = df_modelo['fecha_pedido'].dt.dayofweek
df_modelo['mes'] = df_modelo['fecha_pedido'].dt.month

# Variables que SÍ están disponibles al momento de predecir
features = [
    "dia_semana",       # Día de la semana (0-6)
    "mes",              # Mes del año (1-12)
    "festival",         #Si hubo festival
    "lag_1",            # Ventas de ayer
    "lag_7",            # Ventas de hace 7 días
    "media_movil_7"     # Promedio de últimos 7 días (excluyendo hoy)

]

X = df_modelo[features]
y = df_modelo["ventas_por_dia"]

print("VARIABLES DEFINIDAS:")
print("=" * 60)
print(f"Variables predictoras (X): {len(features)}")
for i, f in enumerate(features, 1):
    print(f"   {i}. {f}")
print(f"\nVariable objetivo (y): ventas_dia")
print(f"\nDimensiones:")
print(f"   X: {X.shape}")
print(f"   y: {y.shape}")

In [ ]:
#SPLIT TEMPORAL 80/20 (CRONOLÓGICO)

corte = int(len(df_modelo) * 0.80)

X_train = X.iloc[:corte]
X_test = X.iloc[corte:]

y_train = y.iloc[:corte]
y_test = y.iloc[corte:]

fecha_test = df_modelo["fecha_pedido"].iloc[corte:]

print("SPLIT TEMPORAL 80/20")
print("=" * 60)
print(f"Total de registros: {len(df_modelo)}")
print(f"Entrenamiento: {len(X_train)} registros (80%)")
print(f"Prueba: {len(X_test)} registros (20%)")

print(f"\nRango de ENTRENAMIENTO:")
print(f"   Desde: {df_modelo['fecha_pedido'].iloc[0]}")
print(f"   Hasta: {df_modelo['fecha_pedido'].iloc[corte-1]}")

print(f"\nRango de PRUEBA:")
print(f"   Desde: {df_modelo['fecha_pedido'].iloc[corte]}")
print(f"   Hasta: {df_modelo['fecha_pedido'].iloc[-1]}")

# Verificar que no hay solapamiento
print(f"\nVERIFICACIÓN:")
print(f"   Última fecha entrenamiento: {df_modelo['fecha_pedido'].iloc[corte-1]}")
print(f"   Primera fecha prueba: {df_modelo['fecha_pedido'].iloc[corte]}")
print(f"   ¿Prueba después de entrenamiento? {df_modelo['fecha_pedido'].iloc[corte] > df_modelo['fecha_pedido'].iloc[corte-1]}")

In [ ]:
# CONSTRUIR BASELINE
# Estrategia: predecir siempre la media del conjunto de entrenamiento

baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)

pred_baseline = baseline.predict(X_test)

# Calcular métricas del baseline
mae_base = mean_absolute_error(y_test, pred_baseline)
rmse_base = mean_squared_error(y_test, pred_baseline) ** 0.5
r2_base = r2_score(y_test, pred_baseline)

print("BASELINE (Media del entrenamiento)")
print("=" * 60)
print(f"Media del entrenamiento: ${y_train.mean():,.2f}")
print(f"\nMÉTRICAS DEL BASELINE:")
print(f"   MAE:  ${mae_base:,.2f}")
print(f"   RMSE: ${rmse_base:,.2f}")
print(f"   R²:   {r2_base:.4f}") #Si es negativo vamos bien.

In [ ]:
#ENTRENAR REGRESIÓN LINEAL
modelo = LinearRegression()
modelo.fit(X_train, y_train)

# Predecir sobre el conjunto de prueba
pred = modelo.predict(X_test)

# Calcular métricas
mae = mean_absolute_error(y_test, pred)
rmse = mean_squared_error(y_test, pred) ** 0.5
r2 = r2_score(y_test, pred)

print("REGRESIÓN LINEAL")
print("=" * 60)
print(f"\nMÉTRICAS DEL MODELO:")
print(f"   MAE:  ${mae:,.2f}")
print(f"   RMSE: ${rmse:,.2f}")
print(f"   R²:   {r2:.4f}") #

In [ ]:
# TABLA COMPARATIVA: BASELINE vs MODELO

comparacion = pd.DataFrame({
    "Modelo": ["Baseline (media)", "Regresión Lineal"],
    "MAE": [mae_base, mae],
    "RMSE": [rmse_base, rmse],
    "R²": [r2_base, r2]
})

print("COMPARACIÓN BASELINE vs MODELO:")
print("=" * 70)
print(comparacion.to_string(index=False))

# Calcular mejora
mejora_mae = ((mae_base - mae) / mae_base) * 100
mejora_rmse = ((rmse_base - rmse) / rmse_base) * 100

print(f"\nMEJORAS DEL MODELO vs BASELINE:")
print(f"   MAE:  {mejora_mae:.1f}% de mejora")
print(f"   RMSE: {mejora_rmse:.1f}% de mejora")

In [ ]:
# COEFICIENTES DEL MODELO

coeficientes = pd.DataFrame({
    "variable": features,
    "coeficiente": modelo.coef_
}).sort_values("coeficiente", key=abs, ascending=False)

print("COEFICIENTES DEL MODELO:")
print("=" * 60)
print(coeficientes.to_string(index=False))

print(f"\nIntercepto (b0): ${modelo.intercept_:,.2f}")
print("\nINTERPRETACIÓN:")
print("   • Coeficiente positivo: la variable aumenta las ventas")
print("   • Coeficiente negativo: la variable disminuye las ventas")
print("   • El valor indica cuánto cambia ventas_dia por cada unidad de la variable")

In [ ]:
# GRÁFICO: Real vs Predicción (temporal)

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(fecha_test, y_test.values,
        label='Real', color='#2c7bb6', linewidth=2, marker='o', markersize=4)
ax.plot(fecha_test, pred,
        label='Predicción', color='#e74c3c', linewidth=2,
        linestyle='--', marker='s', markersize=4)

ax.set_title('Ventas Reales vs Predicción (Conjunto de Prueba)',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Fecha', fontsize=11)
ax.set_ylabel('Ventas ($)', fontsize=11)
ax.legend(loc='upper left', fontsize=11)
ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

print("OBSERVACIONES:")
print(f"   • El modelo sigue la tendencia general")
print(f"   • Hay desviaciones en días específicos")
print(f"   • Los errores grandes podrían investigarse")

In [ ]:
# GRÁFICO: Real vs Predicho (dispersión)

fig, ax = plt.subplots(figsize=(7, 7))

ax.scatter(y_test, pred, alpha=0.6, color='#2c7bb6', edgecolors='#1a5490', s=60)

# Línea de referencia ideal (y = x)
lim_min = min(y_test.min(), pred.min()) * 0.95
lim_max = max(y_test.max(), pred.max()) * 1.05
ax.plot([lim_min, lim_max], [lim_min, lim_max],
        color='#e74c3c', linestyle='--', linewidth=2, label='Predicción perfecta')

ax.set_xlabel('Valor Real ($)', fontsize=11)
ax.set_ylabel('Valor Predicho ($)', fontsize=11)
ax.set_title('Real vs Predicho', fontsize=14, fontweight='bold', pad=15)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

print("OBSERVACIONES:")
print("   • Si los puntos están sobre la línea roja: predicción perfecta")
print("   • Si están dispersos: mayor error")
print("   • Si hay patrón: el modelo tiene sesgo")

In [ ]:
# ANÁLISIS DE RESIDUOS

resultados = pd.DataFrame({
    "fecha": fecha_test.values,
    "real": y_test.values,
    "prediccion": pred
})
resultados["residuo"] = resultados["real"] - resultados["prediccion"]
resultados["error_abs"] = resultados["residuo"].abs()

print("PRIMEROS 10 RESIDUOS:")
print(resultados.head(10).to_string(index=False))

print(f"\nESTADÍSTICAS DE RESIDUOS:")
print(f"   Residuo promedio: ${resultados['residuo'].mean():,.2f}")
print(f"   Residuo mínimo: ${resultados['residuo'].min():,.2f}")
print(f"   Residuo máximo: ${resultados['residuo'].max():,.2f}")

print(f"\nTOP 5 ERRORES MÁS GRANDES:")
print(resultados.nlargest(5, 'error_abs').to_string(index=False))

In [ ]:
# GUARDAR RESULTADOS
from pathlib import Path

# Guardar predicciones
ruta_pred = BASE_DIR / DATA_PROCESSED / 'predicciones_proyecto_guia08.csv'
resultados.to_csv(ruta_pred, index=False, encoding='utf-8')
print(f"Predicciones guardadas: {ruta_pred.name}")

# Guardar métricas
ruta_metricas = BASE_DIR / DATA_DOCS / 'metricas_proyecto_guia08.csv'
comparacion.to_csv(ruta_metricas, index=False, encoding='utf-8')
print(f"Métricas guardadas: {ruta_metricas.name}")
